# MCP Hibrido: SQL + Vector DB

                 LLM
                  │
                  ▼
             MCP Client
                  │
                  ▼
             MCP Server
        ┌─────────┴─────────┐
        ▼                   ▼
    SQL Tools         Vector Tools
        │                   │
        ▼                   ▼
       PostgreSQL          ChromaDB


In [24]:
# Basic imports
import psycopg2, chromadb, json
from pathlib import Path

# Langchain imports
from langchain_ollama import OllamaEmbeddings
from fastmcp import FastMCP

# **Database**

In [14]:
conn = psycopg2.connect(
    host="localhost",
    database="WorldCup",
    user="guane",
    password="tu_password",
    port="5432"
)

print("Conectado")

Conectado


# **Vector database**

In [20]:
PROJECT_ROOT = Path.cwd().parents[1]
VECTOR_DB_PATH = PROJECT_ROOT / "data" / "vector_db_2"
COLLECTION_NAME = "mundiales_football"

client = chromadb.PersistentClient(path=str(VECTOR_DB_PATH))
collections = [c.name for c in client.list_collections()]

print("collections:", collections)


collections: ['mundiales_football']


# **SQL + Vector**

In [22]:
EMBEDDING_MODEL = "nomic-embed-text:latest"
collection = client.get_collection(COLLECTION_NAME)
embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)

In [25]:
def sql_matches_by_year(year: int, limit: int = 10):
    query = """
    SELECT year, datetime, hometeamname, hometeamgoals, awayteamgoals, awayteamname
    FROM world_cup_matches_raw
    WHERE year = %s
    ORDER BY datetime
    LIMIT %s
    """
    with conn.cursor() as cursor:
        cursor.execute(query, (str(year), limit))
        rows = cursor.fetchall()

    columns = [
        "year",
        "datetime",
        "home_team",
        "home_goals",
        "away_goals",
        "away_team",
    ]
    return [dict(zip(columns, row)) for row in rows]


def vector_search_worldcup(question: str, n_results: int = 3):
    query_embedding = embeddings.embed_query(question)
    result = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
    )
    docs = result.get("documents", [[]])[0]
    metas = result.get("metadatas", [[]])[0]

    return [
        {
            "rank": idx + 1,
            "text": doc,
            "metadata": meta or {},
        }
        for idx, (doc, meta) in enumerate(zip(docs, metas))
    ]


print("Functions:")
print("- sql_matches_by_year(year, limit)")
print("- vector_search_worldcup(question, n_results)")
print(f"- embedding_model: {EMBEDDING_MODEL}")

Functions:
- sql_matches_by_year(year, limit)
- vector_search_worldcup(question, n_results)
- embedding_model: nomic-embed-text:latest


# **Functions as tools MCP**

We create a server MCP with 2 tools:
- A for SQL
- A for Vector DB

In [26]:
mcp = FastMCP("WorldCup Hybrid MCP")


@mcp.tool()
def get_matches_by_year(year: int, limit: int = 10):
    """Devuelve partidos del Mundial para un anio."""
    return sql_matches_by_year(year=year, limit=limit)


@mcp.tool()
def search_worldcup_context(question: str, n_results: int = 3):
    """Busca contexto historico del Mundial en la base vectorial."""
    return vector_search_worldcup(question=question, n_results=n_results)


print("Tools MCP registradas:")
for tool_name in ["get_matches_by_year", "search_worldcup_context"]:
    print(f"- {tool_name}")

print("\nPara correr el servidor MCP en terminal:")
print("python src/proofs/worldcup_mcp.py")

Tools MCP registradas:
- get_matches_by_year
- search_worldcup_context

Para correr el servidor MCP en terminal:
python src/proofs/worldcup_mcp.py


# **Proofs tools (modo notebook)**

In [27]:
print("Prueba SQL tool:")
print(json.dumps(get_matches_by_year(year=1930, limit=3), indent=2, ensure_ascii=False))

print("\nPrueba Vector tool:")
print(json.dumps(search_worldcup_context("primer mundial de futbol", n_results=2), indent=2, ensure_ascii=False))

Prueba SQL tool:
[
  {
    "year": "1930",
    "datetime": "13 Jul 1930 - 15:00 ",
    "home_team": "USA",
    "home_goals": "3",
    "away_goals": "0",
    "away_team": "Belgium"
  },
  {
    "year": "1930",
    "datetime": "13 Jul 1930 - 15:00 ",
    "home_team": "France",
    "home_goals": "4",
    "away_goals": "1",
    "away_team": "Mexico"
  },
  {
    "year": "1930",
    "datetime": "14 Jul 1930 - 12:45 ",
    "home_team": "Yugoslavia",
    "home_goals": "2",
    "away_goals": "1",
    "away_team": "Brazil"
  }
]

Prueba Vector tool:
[
  {
    "rank": 1,
    "text": "Historia  de  los  Mundiales  de  Fútbol  \nUn  análisis  exhaustivo  de  la  evolución  técnica,  social  y  global  del  torneo  de  la  FIFA  \n \nPágina  1  -  Introducción:  El  fenómeno  social  y  deportivo  global  \nLa  Copa  Mundial  de  la  FIFA  constituye,  sin  lugar  a  dudas,  el  acontecimiento  deportivo  y  cultural  \nde\n \nmayor\n \nenvergadura\n \ny\n \nresonancia\n \nen\n \nel\n \nplaneta\n \

# **Paso 6: Router con LLM (decide SQL o Vector)**

En esta parte, un LLM clasifica cada pregunta y decide a que tool enviar la consulta:
- `sql` -> `get_matches_by_year`
- `vector` -> `search_worldcup_context`

In [ ]:
import re
from langchain_ollama import ChatOllama

ROUTER_MODEL = "llama3.2:3b"
router_llm = ChatOllama(model=ROUTER_MODEL, temperature=0)


def extract_year(text: str):
    match = re.search(r"(19|20)\d{2}", text or "")
    return int(match.group(0)) if match else None


def route_with_llm(question: str) -> str:
    prompt = f"""
Eres un clasificador de rutas para un sistema MCP de Mundiales.
Responde SOLO con una palabra: sql o vector.

Usa:
- sql: cuando pidan partidos por anio o resultados estructurados por anio.
- vector: cuando pidan contexto historico, explicaciones o informacion semantica.

Pregunta: {question}
Ruta:
""".strip()

    decision = (router_llm.invoke(prompt).content or "").strip().lower()
    if "sql" in decision:
        return "sql"
    return "vector"


def answer_with_router(question: str, limit: int = 5, n_results: int = 3):
    route = route_with_llm(question)

    if route == "sql":
        year = extract_year(question)
        if year is not None:
            data = get_matches_by_year(year=year, limit=limit)
            return {"route": "sql", "question": question, "data": data}
        # Si no hay anio, redirigir a vector
        route = "vector"

    data = search_worldcup_context(question=question, n_results=n_results)
    return {"route": route, "question": question, "data": data}


# Demo de ruteo
examples = [
    "Muestrame los partidos del anio 1930",
    "Explica por que el Mundial de 1930 fue importante",
    "Resultados del mundial 2014",
    "Como evoluciono tacticamente el futbol en los mundiales",
]

for q in examples:
    result = answer_with_router(q)
    print(f"\nPregunta: {q}")
    print(f"Ruta elegida: {result['route']}")
    print("Preview:", json.dumps(result["data"][:1], ensure_ascii=False, indent=2))

In [19]:
# Cerrar conexion SQL al finalizar (opcional)
# conn.close()